In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np

In [ ]:
df = sns.load_dataset("diamonds")

In [ ]:
print(df.head())

In [ ]:
print(f"Total Rows: {df.shape[0]} & Columns: {df.shape[1]}")
#Total Rows: 53940 & Columns: 10

In [ ]:
columns = ["carat", "price"]

In [ ]:
iqr_mask = pd.Series(False, index=df.index)

for col in columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    iqr_mask |= (df[col] < lower) | (df[col] > upper)

In [ ]:
z_mask = pd.Series(False, index=df.index)

for i in columns:
    z = (df[i] - df[i].mean()) / df[i].std()
    z_mask |= z.abs() > 3

In [ ]:
print("IQR outliers:", iqr_mask.sum()) #IQR outliers: 6416
print("Z-score outliers:", z_mask.sum()) #Z-score outliers: 2350

In [ ]:
both = iqr_mask & z_mask
iqr_only = iqr_mask & ~z_mask
z_only = z_mask & ~iqr_mask
neither = ~iqr_mask & ~z_mask

print("\nOverlap (both methods):", both.sum())
print("Only IQR:", iqr_only.sum())
print("Only Z-score:", z_only.sum())
print("Neither:", neither.sum())

In [ ]:
df_compare = df.copy()
df_compare["IQR_outlier"] = iqr_mask
df_compare["Z_outlier"] = z_mask

In [ ]:
print("Disagreements:")
print(df_compare[iqr_only | z_only][["carat", "price", "IQR_outlier", "Z_outlier"]].head(20))
print(df_compare[iqr_only | z_only][["carat", "price", "IQR_outlier", "Z_outlier"]].tail(20))

In [ ]:
df["log_price"] = np.log(df["price"])

In [ ]:
print(df["price"].skew()) #1.618395283383529
print(df["log_price"].skew()) #0.11529585821715065

In [ ]:
z_mask_lprice = pd.Series(False, index=df.index)
z_lprice = (df["log_price"] - df["log_price"].mean()) / df["log_price"].std()
z_mask_lprice |= z_lprice.abs() > 3

print("Z-score outliers after log transformation:", z_mask_lprice.sum()) #Z-score outliers after log transformation: 0

In [ ]:
print("Final Conclusion: No Outliers Detected")

In [ ]:
df.to_csv(r"diamonds.csv", index = False)